# Tutorial: how to use `pangeo-fish`


**Overview.**

This Jupyter notebook demonstrates how to use `pangeo-fish`.

Specifically, we will fit the geolocation on the data from the study conducted by M. Gonze et al. titled "Combining acoustic telemetry with archival tagging to investigate the spatial dynamics of the understudied pollack *Pollachius pollachius*", accepted for publication in the Journal of Fish Biology.

We will use the biologging tag "A19124", which was attached to pollack fish.

As for the reference Earth Observation (EO) data, we consider the European Union Copernicus Marine Service Information (CMEMS) product "NORTHWESTSHELF_ANALYSIS_FORECAST_PHY_004_013".

_NB: In addition to the Data Storage Tag (DST), the biologging data includes **teledetection by acoustic signals**, as well as the release and recapture/death information of the fish._

Both the reference EO and the biologging data are publicly available, and the computations should be tractable for most standard laptops.

**Workflow.**

Let's first summarize the key steps for running the geolocation:

1. **Define the configuration:** define the required parameters for the analysis.
2. **Compare the reference data with the DST information:** compare the data from the reference model with the biologging data. 
3. **Regrid the comparison to HEALPix:** translate the comparison into a HEALPix grid to avoid spatial distortion.
4. **Construct the temporal emission matrix:** create a temporal emission probability distribution (_pdf_) from the transformed grid.
5. **Construct another emission matrix with the acoustic detections:** calculate a similar model to the previous one, using this time the acoustic teledetections.
6. **Combine and normalize the matrices:** merge and normalize the two _pdfs_.
7. **Estimate (or _fit_) the geolocation model:** determine the parameters of the model based on the normalized emission matrix.
8. **Compute the state probabilities and generate trajectories:** compute the fish's location probability distribution and generate subsequent trajectories.
9. **Visualization:** visualize the evolution of the spatial probabilities over time and export the video.

Throughout this tutorial, you will gain practical experience in setting up and executing a typical workflow using `pangeo-fish` such that you can then apply the tool with your use-case study.

## 1. Initialization and configuration definition

In this step, we prepare the execution of the analysis.
It includes:
- Installing the necessary packages.
- Importing the required libraries.
- Defining the parameters for the next stages of the workflow.
- Configuring the cluster for distributed computing.
    

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

sys.path.append("../")
import pangeo_fish

In [ ]:
#### tag_name corresponds to the name of the biologging tag name (DST identification number),
# which is also a path for storing all the information for the specific fish tagged with tag_name.
# tag_name = "SV_A18991_V2"

# # tag_root specifies the root URL for tag data used for this computation.
# tag_root = "donnees_importees"

tag_name = "A19124"

# tag_root specifies the root URL for tag data used for this computation.
tag_root = "https://data-taos.ifremer.fr/data_tmp/cleaned/tag/"


# ref_url is the path to the reference model
ref_url = "https://data-taos.ifremer.fr/kerchunk/copernicus_model_NWSHELF_2022.json"

# scratch_root specifies the root directory for storing output files.
# storage_options specifies options for the filesystem storing output files.

## example for using your local file system instead
scratch_root = "./test/"
storage_options = None

# Default chunk value for time dimension.  This values depends on the configuration of your dask cluster.
chunk_time = 4

# Either to use a HEALPix grid (["cells"]) or a 2D grid (["x", "y"])
dims = ["cells"]

# bbox, bounding box, defines the latitude and longitude range for the analysis area.
bbox = {"latitude": [45, 60], "longitude": [-10, 10]}
bbox = {"latitude": [46, 60], "longitude": [-20, 5]}
# relative_depth_threshold defines the acceptable fish depth relative to the maximum tag depth.
# It determines whether the fish can be considered to be in a certain location based on depth.
relative_depth_threshold = 0

# refinement level  defines the resolution of the healpix grid used for regridding.
refinement_level = 8  # int(log2(4098))

# min_vertices sets the minimum number of vertices for a valid transcription for regridding.
min_vertices = 1

# differences_std sets the standard deviation for scipy.stats.norm.pdf.
# It expresses the estimated certainty of the field of difference.
differences_std = 0.75
# initial_std sets the covariance for initial event.
# It shows the certainty of the initial area.
initial_std = 1e-6 if dims == ["x", "y"] else 1e-5
# recapture_std sets the covariance for recapture event.
# It shows the certainty of the final recapture area if it is known.
recapture_std = 1e-3  # 1e-4
# earth_radius defines the radius of the Earth used for distance calculations.
earth_radius = ureg.Quantity(6371, "km")
# maximum_speed sets the maximum allowable speed for the tagged fish.
maximum_speed = ureg.Quantity(60, "km / day")
# adjustment_factor adjusts parameters for a more fuzzy search.
# It will factor the allowed maximum displacement of the fish.
adjustment_factor = 5
# truncate sets the truncating factor for computed maximum allowed sigma for convolution process.
truncate = 4

# receiver_buffer sets the maximum allowed detection distance for acoustic receivers.
receiver_buffer = ureg.Quantity(1000, "m")


# tolerance describes the tolerance level of the search during the fitting/optimization of the geolocation.
# Smaller values will make the optimization iterate more
tolerance = 1e-3 if dims == ["x", "y"] else 1e-6

# track_modes defines the modes for generating fish's trajectories.
track_modes = ["mean", "mode"]
# additional_track_quantities sets quantities to compute for tracks using moving pandas.
additional_track_quantities = ["speed", "distance"]


# time_step defines the time interval between each frame of the visualization
time_step = 3

In [ ]:
# Define target root directories for storing analysis results.
target_root = f"{scratch_root}/{tag_name}"

# Defines default chunk size for optimization.
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

Now that everything is set up, we can start by loading the biologging data (or _tag_)

In [ ]:
from pangeo_fish.helpers import open_tag, to_time_slice

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=storage_options
)
tag

You can plot the time series of the DST with the function `plot_tag()`:

In [ ]:
from pangeo_fish.helpers import plot_tag

plot = plot_tag(
    tag=tag,
    tag_log=tag_log,
    # you can directly save the plot if you want
    save_html=True,
    storage_options=storage_options,
    target_root=target_root,
)
plot

## 2. Compare the reference data with the DST logs

In this step, we compare the reference model data with Data Storage Tag information.
The process involves reading and cleaning the reference model, aligning time, converting depth units and subtracting the tag data from the model.
We also illustrate how to plot and saving the result.

## 2.bis Compare the reference data with the DST logs (using copernicus)

In [ ]:
import copernicusmarine
import xarray as xr
from pangeo_fish.io import prepare_dataset
import pandas as pd

Username = "cchauveau1"
Password = "C0pernicus-st@giaiRe!"

ds_thetao_zos = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "zos"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
    start_datetime=pd.Timestamp(time_slice.start.item()).strftime("%Y-%m-%d"),
    end_datetime=pd.Timestamp(time_slice.stop.item()).strftime("%Y-%m-%d"),
    username=Username,
    password=Password,
)

static_var = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_static",
    variables=["deptho", "mask"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
    username=Username,
    password=Password,
)

static_var = static_var.assign_coords(longitude=ds_thetao_zos["longitude"].values)

ds_all = xr.merge([ds_thetao_zos, static_var], compat="no_conflicts")
ds_all = ds_all.rename({"latitude": "lat", "longitude": "lon"})
model = prepare_dataset(ds_all)
model

In [ ]:
import pandas as pd
from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.tags import adapt_model_time, reshape_by_bins, to_time_slice

# Only in case of days
tag_log["time"] = tag_log["time"].dt.floor("D")

start = pd.Timestamp(tag_log["time"].min().item())
end = pd.Timestamp(tag_log["time"].max().item())

extended_time_slice = slice(start, end)

reference_model = (
    model.sel(time=adapt_model_time(extended_time_slice))
    .sel(lat=slice(*bbox["latitude"]), lon=slice(*bbox["longitude"]))
    .pipe(
        lambda ds: ds.sel(
            depth=slice(None, (tag_log["pressure"].max() + 100).compute())
        )
    )
).chunk(
    {"time": chunk_time, "depth": -1}
)  # .chunk({"time": chunk_time, "depth": -1, "lat": 100, "lon": 100})


reference_model

In [ ]:
%%time
# Reshape the tag log, so that it bins to the time step of reference_model
reshaped_tag = reshape_by_bins(
    tag_log,
    dim="time",
    bins=(
        reference_model.cf.add_bounds(["time"], output_dim="bounds")
        .pipe(bounds_to_bins, bounds_dim="bounds")
        .get("time_bins")
    ),
    other_dim="obs",
).chunk({"time": chunk_time})

reshaped_tag

In [ ]:
chunk_time = 1
from pangeo_fish.helpers import load_model, compute_diff

diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
)[0]

In [ ]:
%%time
diff.compute()

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).plot()
diff

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. HEALPix regridding

In this step, we regrid the data from above to HEALPix coordinates. HEALPix (Hierarchical Equal Area isoLatitude Pixelization) is a spherical pixelation scheme that divides the sphere into equal-area pixels, eliminating spatial distortion [see the HEALPix documentation](https://healpix.sourceforge.io/).

This is a complex process, composed of several steps such as defining the HEALPix grid, creating the target grid and computing interpolation weights

Fortunately though, `pangeo-fish` embarks high-level functions to do the work for us!

In [ ]:
from pangeo_fish.helpers import open_diff_dataset, regrid_dataset

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped = regrid_dataset(
    ds=diff, refinement_level=refinement_level, min_vertices=min_vertices, dims=dims
)[0]
reshaped

Let's plot the same chart as before to check that the HEALPix regridding hasn't changed the data

In [ ]:
reshaped["diff"].count(dims).plot()

In [ ]:
# Saves the result
reshaped.chunk(default_chunk_dims).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
import xdggs

reshaped["diff"].dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).compute().dggs.explore()

## 4. Compute the emission probability distribution

In this step, we use the comparison result from the step above to construct the emission probability matrix.

This comparison is essentially he differences between the temperature measured by the tag and the reference sea temperature. 

The emission probability matrix represents the likelihood of observing a specific temperature difference given the model parameters and configurations.

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",  # diff-regridded.zarr,
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)  # .pipe(lambda ds: ds.merge(ds[["latitude", "longitude"]].compute()))
# ... and compute the emission matrices
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

Whatever the temporal distribution looks like, they must **never** (i.e, at _any time step_) sum to 0.

How could we check that visually? You'd have guessed it by now: similarly as before!

In [ ]:
emission_pdf["pdf"].dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).compute().dggs.explore()

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
emission_pdf["pdf"].count(dims).plot()

In [ ]:
# Save the dataset
emission_pdf.to_zarr(
    f"{target_root}/emission.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. Compute a 2nd _pdf_ with the acoustic detections

In this step, the goal is to calculate another emission distribution, this time from the acoustic detections.
**As such, it requires the tag to include at least one detection.**

These additional probabilities will enhance the emission _pdf_ constructed in the previous step by incorporating information from acoustic telemetry.

_NB: we will merge and normalize the two pdfs in the next stage of the workflow._

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# Load the previous emission pdf and compute the emission probabilities based on acoustic detections
emission_pdf = xr.open_dataset(
    f"{target_root}/emission.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)  # chunk?
emission_pdf

In [ ]:
acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_pdf,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
import xdggs

acoustic_pdf

If you wonder how this emission matrix looks like, you can plot a combined plot of the detections and the probabilities:

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

### Explanations
On the plot above, at detection times the number of counted values drop to a few value (`5` in this example).

These numbers correspond to the number of pixels that covers the detection area.

Therefore, such drop is expected, since at those times we know that the fish was detected around the acoustic receivers, and so it **can't** be elsewhere.

These sporadic detections will constraint a lot the geolocation model upon optimizing!

**The next cell is optional. It will save the acoustic emission distribution. It is not necessary (see the next step).**

In [ ]:
acoustic_pdf.to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
)

## 5.2. Compute a 3rd pdf with bathy

In this step, the goal is to calculate another emission distribution, this time using a bathymetry like GEBCO, EMODNET...

These additional probabilities will enhance the emission pdf constructed in the previous step by incorporating information from bathymetry information.

/!\ nécessite d'avoir ouvert le ref model et d'avoir diff sous la main

In [ ]:
emission_pdf = xr.open_dataset(
    f"{target_root}/emission.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
emission_pdf

In [ ]:
import fsspec
import numpy as np
from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.tags import adapt_model_time, reshape_by_bins, to_time_slice
from pangeo_fish.bathy import batch_compute_pdf_bathy
import healpy as hp
from pangeo_fish.bathy import (
    compute_healpix_histogram_region_bin_size,
)

# bathy_url is the path to the bathymetry file
bathy_url = "https://data-taos.ifremer.fr/global_bathy/GEBCO_2024.zarr"

nside = 256

Importation de la base de donnée de bathymétrie

In [ ]:
full_bathy = xr.open_dataset(
    "s3://gfts-reference-data/gebco_2024_new.zarr",
    engine="zarr",
    chunks={},
    storage_options={
        "profile": "gfts",
        "client_kwargs": {"endpoint_url": "https://s3.gra.perf.cloud.ovh.net"},
    },
).rename({"lat": "latitude", "lon": "longitude"})

subset_bathy = full_bathy.sel(
    {dim: slice(bounds[0], bounds[1]) for dim, bounds in bbox.items()}
)

In [ ]:
ds_histo = compute_healpix_histogram_region_bin_size(
    subset_bathy,
    nside=nside,
    max_depth_m=1000,  # <- profondeur max désirée en mètres
    depth_bin_size=16,  # <- largeur d’un bin en mètres
)

hist_ids = ds_histo.cell_ids.values  # cell ids dans ton ds_histo
pdf_ids = emission_pdf.cell_ids.values  # cell ids dans emission_pdf

common = np.intersect1d(hist_ids, pdf_ids)
# isel avec masque
mask = np.isin(hist_ids, common)
ds_histo.isel(cells=np.where(mask)[0])

reshaped_tag = reshape_by_bins(
    tag_log,
    dim="time",
    bins=(
        reference_model.cf.add_bounds(["time"], output_dim="bounds")
        .pipe(bounds_to_bins, bounds_dim="bounds")
        .get("time_bins")
    ),
    other_dim="obs",
).chunk({"time": chunk_time})

pdf_da_func = batch_compute_pdf_bathy(
    ds_histo,
    reshaped_tag,
    target_root,
    batch_size=10000,
)
sum_over_cells = pdf_da_func.sum(dim="cells", skipna=True)

bathy_pdf = pdf_da_func / sum_over_cells

Enregistre la pdf de bathy seule (si besoin est de la réutiliser indépendamment des autres)

In [ ]:
bathy_pdf.compute().to_zarr(
    f"{target_root}/bathy_pdf_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_version=2,
    storage_options=storage_options,
)

In [ ]:
bathy_pdf.pdf_bathy.compute().dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8, cmap="viridis")

Cette cellule ajoute la pdf calculée à d'autres, elle nécessite d'en avoir d'autres...

In [ ]:
emission_with_bathy = emission_pdf.merge(
    bathy_pdf.drop_indexes(["cell_ids"]), compat="override"
)
emission_with_bathy

## 5.3. Compute a 4th pdf with tide


In [ ]:
from Tide_Hue.tide_function_Copy1 import *

In [ ]:
import xarray as xr
import pandas as pd
from ipywidgets import interact, IntSlider
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from numpy.linalg import lstsq
from tqdm import tqdm
from scipy.signal import savgol_filter
import xdggs
import sparse

In [ ]:
tag_root = "https://data-taos.ifremer.fr/data_tmp/cleaned/tag/"
tag_name = "A19124"
# emplacement à CHANGER selon local ou cloud
dims = ["cells"]
tag, tag_log, time_slice = load_tag(
    tag_root=tag_root,
    tag_name=tag_name,
    storage_options=None,
)
tag

In [ ]:
ds = tag_log["pressure"]
df_resample = ds.resample(
    {"time": "10min"}
).first()  # On prends toutes les 10 minutes la pression
df_resample = df_resample.reset_index("time")
df = df_resample.to_dataframe()
df = df.reset_index()  # On enlève l'index time
df["time"] = (
    tag_log["time"].resample({"time": "10min"}).first()
)  # <--- Ajout de la colonne timedans df + on prends toute les 10 minutes

In [ ]:
tag_test = tide_behav_extr(
    df=df,
    tagno=tag_name,
    tideFL=1,
    tideLV=[0.42, 0.85, 0.6],
    behavFL=16,
    behavLV=[0.42, 0.85, 0.6],
    dt=1,
)

tag_test

In [ ]:
# Dataset : ds
time = tag_test["time"].values
depth = tag_test["depth"].values
tide_found = tag_test["tide_found"].values  # 0 ou 1

# Création du plot
plt.figure(figsize=(15, 4))
plt.plot(time, depth, color="blue", label="Profondeur")

# Ajouter un fond vert pour les périodes de marée
# On va utiliser fill_between pour les zones où tide_found==1
plt.fill_between(
    time,
    depth.min(),
    depth.max(),
    where=(tide_found == 1),
    color="green",
    alpha=0.3,
    label="Marée détectée",
)

plt.xlabel("Temps")
plt.ylabel("Profondeur (m)")
plt.title(f'Temporal Depth & Tidal Detection - Tag {tag_test.attrs["tagno"]}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
tag_test.to_zarr(f"data/{tag_name}/tide_behav.zarr", mode="w")

In [ ]:
tide_behav = xr.open_dataset(f"data/{tag_name}/tide_behav.zarr", engine="zarr")
tide_behav

In [ ]:
data_360 = xr.open_dataset("donnees_importees/Subset_NS_EC_forTMD30_21_2_2025.nc")

data = convert_lon_360_to_180(data_360, bbox)
data

In [ ]:
from Tide_Hue.tide_function_Copy1 import *

ds_pdf = tide_pdf(tide_behav, data)
ds_pdf

In [ ]:
emission_pdf = ds_pdf
# Save in zarr
emission_pdf.to_zarr(f"data/{tag_name}/tide_pdf.zarr", mode="w")

In [ ]:
import xarray as xr

LIK = xr.open_dataset(f"data/{tag_name}/tide_pdf.zarr", engine="zarr", chunks={})
LIK = LIK.rename({"mask": "ocean_mask"})
print(type(LIK["ocean_mask"].data))
LIK

In [ ]:
from pangeo_fish.helpers import open_diff_dataset, regrid_dataset

refinement_level = 8

reshaped = regrid_dataset(
    ds=LIK, refinement_level=refinement_level, min_vertices=1, dims=["cells"]
)[0]
reshaped

In [ ]:
# differences_std sets the standard deviation for scipy.stats.norm.pdf.
# It expresses the estimated certainty of the field of difference.
differences_std = 0.75
# initial_std sets the covariance for initial event.
# It shows the certainty of the initial area.
initial_std = 1e-4  # if dims == ["x", "y"] else 1e-5
# recapture_std sets the covariance for recapture event.
# It shows the certainty of the final recapture area if it is known.
recapture_std = 5e-4  # 1e-4

In [ ]:
from pangeo_fish.helpers import to_healpix
import pangeo_fish.distributions as distrib

dims = ["cells"]
diff_ds = LIK
events_ds = tag["tagging_events"].ds
differences_std = differences_std
grid = reshaped[["latitude", "longitude"]].compute()
initial_position = events_ds.sel(event_name="release")
final_position = events_ds.sel(event_name="fish_death")
grid = to_healpix(grid)

In [ ]:
initial_std = 1e-4
recapture_std = 5e-4
initial_probability = distrib.healpix.normal_at(
    grid, pos=initial_position, sigma=initial_std
)
final_probability = distrib.healpix.normal_at(
    grid, pos=final_position, sigma=recapture_std
)

In [ ]:
from toolz.dicttoolz import valfilter

reshaped2 = reshaped.assign(
    valfilter(
        lambda x: x is not None,
        {
            "initial": initial_probability,
            "final": final_probability,
        },
    )
)
print(reshaped)
reshaped2

In [ ]:
# Permet d'éviter des problèmes au moment du save / compute
var = reshaped2["ocean_mask"].compute()
if isinstance(var.data, sparse.SparseArray):
    reshaped2["ocean_mask"] = var.copy(data=var.data.todense())
reshaped = reshaped2

In [ ]:
ds = reshaped  # ton dataset
pdf = ds.pdf
# 1. Rechunker la variable pdf pour un chunking optimal
#    (chunk sur le temps, taille libre sur les autres dims)
pdf_chunked = ds.pdf.chunk({"time": 1})  # ajuste selon la mémoire disponible

# 2. Construire le graphe de tâches (pas de calcul ici)
delayed_blocks = [
    pdf_chunked.isel(time=slice(start, min(start + 1, pdf_chunked.sizes["time"])))
    for start in range(pdf_chunked.sizes["time"])
]

# 3. Un seul .compute() — Dask distribue sur le cluster
pdf_full = xr.concat(delayed_blocks, dim="time").compute()

# 4. Reconstruire le dataset
ds_full = ds.drop_vars("pdf").assign(pdf=pdf_full)
print("DONE — Dataset computed en parallèle !")

In [ ]:
for name in ["final", "initial"]:
    computed = ds_full[
        name
    ].compute()  # déclenche le calcul -> objet sparse.COO en mémoire
    if hasattr(computed.data, "todense"):
        dense_data = np.asarray(computed.data.todense())
        ds_full[name] = computed.copy(data=dense_data)

ds_full.to_zarr(f"test/{tag_name}/tide_pdf_healpix.zarr", mode="w")

In [ ]:
daily = ds_full.resample(time="1D").mean(skipna=True)
daily

In [ ]:
common_pix = daily.cell_ids.to_index().intersection(
    emission_with_bathy.cell_ids.to_index()
)
common_pix

In [ ]:
ds1_common = daily.where(daily.cell_ids.isin(common_pix).compute(), drop=True)
ds2_common = emission_with_bathy.where(
    emission_with_bathy.cell_ids.isin(common_pix).compute(), drop=True
)

In [ ]:
ds1_common = ds1_common.rename({"pdf": "pdf_tide"})
ds1_common
ds1_common_iniless = ds1_common.drop_vars(["final", "initial"])
ds1_common_iniless

In [ ]:
ds2_common

In [ ]:
emission_with_bathy_tide = ds1_common_iniless.merge(ds2_common, compat="override")
emission_with_bathy_tide

In [ ]:
m1 = (
    emission_with_bathy_tide.pdf.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m2 = (
    emission_with_bathy_tide.pdf_tide.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m3 = (
    emission_with_bathy_tide.pdf_bathy.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)

In [ ]:
import ipywidgets as ipw


def map_with_slider_below(m):
    return ipw.VBox([m.map, m.sliders["time"]])


ipw.HBox(
    [map_with_slider_below(m1), map_with_slider_below(m2), map_with_slider_below(m3)]
)


def extract_slider(map_):
    return map_.sliders["time"]


ipw.jslink((extract_slider(m1), "value"), (extract_slider(m2), "value"))
ipw.jslink((extract_slider(m1), "value"), (extract_slider(m3), "value"))
for m in [m1, m2, m3]:
    m.layout.width = "300px"
    m.layout.height = "400px"
ipw.HBox([m1, m3, m2])

In [ ]:
emission_with_bathy_tide.compute().to_zarr(
    f"{target_root}/tide_bathy_pdf_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_version=2,
    storage_options=storage_options,
)

## 6. Combine and normalize the X distributions

As mentioned before, before fitting the model, we need to merge the `emission` distribution with the `acoustic` one and normalize the result.

The normalization ensures that the probabilities sum up to one for each time step. 

In [ ]:
# La cellule où final == 1
final_cell = emission_pdf["final"].compute().argmax("cells")
print(final_cell.values)

# La valeur du pdf à cet endroit au dernier temps
print(emission_pdf["pdf"].isel(time=-1, cells=final_cell).compute())

In [ ]:
print("Vérifier combien de NaN sur le dernier temps")
print(emission_pdf["pdf"].isel(time=-1).isnull().sum().compute())

In [ ]:
print(" Vérifier si cette cellule a des NaN sur toute la série temporelle")
print(emission_pdf["pdf"].isel(cells=521).isnull().sum().compute())

In [ ]:
# Ou seulement à la fin ?
emission_pdf["pdf"].isel(cells=521).isnull().plot()

## 6.5 Cas ou il n'y a pas d'acoustique

In [ ]:
import inspect
from pangeo_fish.helpers import normalize_pdf
from pangeo_fish.pdf import combine_emission_pdf

print(inspect.getsource(combine_emission_pdf))

In [ ]:
import numpy as np

emission_pdf = xr.open_dataset(
    f"{target_root}/tide_bathy_pdf_{tag_name}.zarr",  # à changer
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
emission_pdf_clean = emission_pdf
# print(emission_pdf["pdf"].isel(time=-1).sum(dims).compute())

In [ ]:
combined = normalize_pdf(
    ds=emission_pdf,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask", "ocean_mask", "tide_found"),
)[0]
combined["pdf"].count(dims).plot()
combined

In [ ]:
m1 = (
    emission_with_bathy_tide.pdf.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m2 = (
    combined.pdf.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)
m3 = (
    emission_with_bathy_tide.pdf_tide.compute()
    .dggs.decode({"grid_name": "healpix", "level": 8, "indexing_scheme": "nested"})
    .dggs.explore(alpha=0.8)
)

In [ ]:
import ipywidgets as ipw


def map_with_slider_below(m):
    return ipw.VBox([m.map, m.sliders["time"]])


ipw.HBox([map_with_slider_below(m1), map_with_slider_below(m2)])
ipw.HBox([map_with_slider_below(m1), map_with_slider_below(m3)])


def extract_slider(map_):
    return map_.sliders["time"]


ipw.jslink((extract_slider(m1), "value"), (extract_slider(m2), "value"))
ipw.jslink((extract_slider(m1), "value"), (extract_slider(m3), "value"))
for m in [m1, m2, m3]:
    m.layout.width = "300px"
    m.layout.height = "400px"
ipw.HBox([m1, m2, m3])

## Enregistrement

In [ ]:
combined2 = combined.drop_vars(["tide_found", "ocean_mask"])
combined2

In [ ]:
combined2.to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

_**Tip:** in case you don't have acoustic detection (or you don't want to use it), you can normalize the `emission_pdf` with:_
```python
from pangeo_fish.helpers import normalize_pdf
combined = normalize_pdf(
    ds=emission_pdf,
    chunks=default_chunk_dims,
    dims=dims,
)[0]
# ... and save the `combined` as shown above
```

**Let's perform a last check before fitting the model's parameters.**

Indeed, remind that the _pdfs_ have been combined as their _product_.

As such, for some time steps, if they don't overlap, the result will be empty (probability of 0 everywhere!).

We can quickly detect if this happens by plotting the sum of the probabilities over the time dimension:

In [ ]:
import xdggs

combined2["pdf"].dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).compute().dggs.explore()

_The sums should equal to `1`._

## 7. Estimate the model's parameters

It is now time to determine the parameters of the model based on the normalized emission matrix.

Precisely, is consists of finding the best `sigma`, which corresponds to the standard deviation of the Brownian motion that models the fish's movement between the time steps.  

To do so, in the following we:
1. Define the lower and upper bounds for `sigma`.  
2. Search for the best `sigma` with `optimize_pdf()`.
3. Save the results of the search (i.e., ` sigma`), along with any additional parameters used during the optimization, a human-readable `.json` file.  

In [ ]:
from pangeo_fish.helpers import optimize_pdf

In [ ]:
# Open the distributions
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
emission

In [ ]:
# Define the parameter's bounds and search for the best value
params = optimize_pdf(
    ds=emission,
    earth_radius=earth_radius,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=maximum_speed,
    tolerance=tolerance,
    dims=dims,
    # the results can be directly saved
    save_parameters=True,
    storage_options=storage_options,
    target_root=target_root,
)
params

## 8. State probabilities and Trajectories

In this second to last step, we calculate the spatial probability distribution (based on the `sigma` found earlier) and further compute trajectories.

_NB: the computation precisely relies on `sigma` and the combined emission pdf._

In [ ]:
from pangeo_fish.helpers import predict_positions

In [ ]:
states, trajectories = predict_positions(
    target_root=target_root,
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

Let's quickly check that the positional probability distribution `states` never sums to 0 for all timesteps!

In [ ]:
(
    states.sum(dims).hvplot(width=500, ylim=(0, 2), title="Sum of the probabilities")
    + states.count(dims).hvplot(width=500, title="Number of none-zero probabilities")
).opts(shared_axes=False)

## 9. Visualization

In this final step, we visualize various aspects of the analysis results to gain insights and interpret the model outcomes. 

We plot the emission distribution, which represents the likelihood of observing a specific temperature difference given the model parameters and configurations. 

Additionally, we visualize the state probabilities, showing the likelihood of the system (i.e, the fish) being in different states (i.e, positions) at each time step. 

We also plot the trajectories decoded before (if you saved them).

They display the possible movement patterns over time. 

Finally, we render the emission matrix and state probabilities in a video and store it.

### 9.1 Plotting the trajectories 

In [ ]:
from pangeo_fish.helpers import plot_trajectories

In [ ]:
states["states"].isel(time=slice(0, 1000, 10)).dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).compute().dggs.explore()

In [ ]:
plot = plot_trajectories(
    target_root=target_root,
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)
plot

### 9.2 Plotting the `states` and `emission` distributions 

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=target_root,
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

The interactive plot above is too large to be stored as a `HMTL` file (as done earlier with the trajectories).

Fortunately, `pangeo-fish` can efficiently render images of `data` and build a video from them! 

In [ ]:
%pip install imageio[ffmpeg]

In [ ]:
video_filename = render_distributions(
    data=data,
    output_path=f"{target_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="images",
    remove_frames=True,
    storage_options=storage_options,
)